# Xây dựng bảng RFM cho khách hàng United Kingdom

Notebook này tạo bảng RFM từ `data/processed/clean_data.csv` cho khách hàng tại United Kingdom trong toàn bộ giai đoạn dữ liệu.

RFM gồm ba chỉ số:

- `Recency`: số ngày kể từ lần mua gần nhất đến ngày tham chiếu.
- `Frequency`: số hóa đơn riêng biệt của khách hàng.
- `Monetary`: tổng số tiền khách hàng đã chi tiêu.

Bảng RFM là đầu vào cho bước EDA RFM và GMM ở phase sau.

## 1. Load dữ liệu sạch

Đầu tiên, đọc `data/processed/clean_data.csv` và chuyển `InvoiceDate` về kiểu thời gian để có thể tính ngày mua gần nhất của từng khách hàng.

Đoạn code dưới đây import thư viện, đọc dữ liệu giao dịch đã làm sạch và parse cột thời gian.

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "clean_data.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "rfm_uk_full.csv"

df = pd.read_csv(DATA_PATH)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

df.head()

## 2. Chọn thị trường United Kingdom

Từ kết quả EDA trước đó, United Kingdom là thị trường chiếm phần lớn doanh thu, khách hàng và số hóa đơn. Vì vậy, bảng RFM được xây dựng riêng cho khách hàng UK để mô hình phân cụm ổn định hơn.

Đoạn code dưới đây lọc dữ liệu UK và tóm tắt lại quy mô của tập dữ liệu sau khi lọc.

In [ ]:
uk_df = df[df["Country"].eq("United Kingdom")].copy()

uk_overview = pd.DataFrame(
    {
        "Metric": [
            "Rows",
            "Customers",
            "Invoices",
            "Revenue",
            "Start Date",
            "End Date",
        ],
        "Value": [
            f"{len(uk_df):,}",
            f"{uk_df['CustomerID'].nunique():,}",
            f"{uk_df['InvoiceNo'].nunique():,}",
            f"£{uk_df['TotalPrice'].sum():,.2f}",
            uk_df["InvoiceDate"].min().strftime("%Y-%m-%d"),
            uk_df["InvoiceDate"].max().strftime("%Y-%m-%d"),
        ],
    }
)

uk_overview

## 3. Xác định snapshot date

`Recency` cần một ngày tham chiếu để đo khoảng cách từ lần mua gần nhất của khách hàng. Cách thường dùng là lấy ngày giao dịch cuối cùng trong dữ liệu cộng thêm một ngày.

Đoạn code dưới đây xác định `snapshot_date`. Với dữ liệu hiện tại, ngày giao dịch cuối cùng là `2011-12-09`, nên ngày tham chiếu là `2011-12-10`.

In [ ]:
last_invoice_date = uk_df["InvoiceDate"].max().normalize()
snapshot_date = last_invoice_date + pd.Timedelta(days=1)

snapshot_summary = pd.DataFrame(
    {
        "Metric": ["Last Invoice Date", "Snapshot Date"],
        "Value": [
            last_invoice_date.strftime("%Y-%m-%d"),
            snapshot_date.strftime("%Y-%m-%d"),
        ],
    }
)

snapshot_summary

## 4. Tính RFM

Dữ liệu giao dịch đang ở cấp độ dòng sản phẩm trong hóa đơn. Ở bước này, dữ liệu được tổng hợp về cấp độ khách hàng, mỗi `CustomerID` chỉ còn một dòng RFM.

Đoạn code dưới đây tính `Recency`, `Frequency` và `Monetary` cho từng khách hàng UK. `Frequency` được tính bằng số `InvoiceNo` riêng biệt, không phải số dòng sản phẩm.

In [ ]:
rfm_uk_full = (
    uk_df.groupby("CustomerID")
    .agg(
        LastPurchaseDate=("InvoiceDate", "max"),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("TotalPrice", "sum"),
    )
    .reset_index()
)

rfm_uk_full["Recency"] = (
    snapshot_date - rfm_uk_full["LastPurchaseDate"].dt.normalize()
).dt.days

rfm_uk_full = rfm_uk_full[["CustomerID", "Recency", "Frequency", "Monetary"]]
rfm_uk_full["Monetary"] = rfm_uk_full["Monetary"].round(2)
rfm_uk_full = rfm_uk_full.sort_values("CustomerID").reset_index(drop=True)

rfm_uk_full.head()

**Nhận xét:** bảng RFM mới dùng toàn bộ giai đoạn dữ liệu của UK. Vì vậy, số khách hàng và khoảng Recency có thể lớn hơn file `rfm_uk.csv` cũ, vốn có dấu hiệu chỉ được tạo từ nửa đầu dữ liệu.

## 5. Kiểm tra bảng RFM

Sau khi tạo RFM, cần kiểm tra lại các công thức chính. Đây là bước quan trọng vì nếu RFM sai thì toàn bộ kết quả phân cụm phía sau cũng sai.

Đoạn code dưới đây kiểm tra số dòng RFM, khoảng giá trị của ba chỉ số và xác nhận bảng không có giá trị thiếu.

In [ ]:
rfm_validation_summary = pd.DataFrame(
    {
        "Metric": [
            "RFM Rows",
            "Distinct UK Customers",
            "Missing Values",
            "Min Recency",
            "Max Recency",
            "Min Frequency",
            "Max Frequency",
            "Min Monetary",
            "Max Monetary",
        ],
        "Value": [
            f"{len(rfm_uk_full):,}",
            f"{uk_df['CustomerID'].nunique():,}",
            f"{int(rfm_uk_full.isna().sum().sum()):,}",
            f"{rfm_uk_full['Recency'].min():,}",
            f"{rfm_uk_full['Recency'].max():,}",
            f"{rfm_uk_full['Frequency'].min():,}",
            f"{rfm_uk_full['Frequency'].max():,}",
            f"£{rfm_uk_full['Monetary'].min():,.2f}",
            f"£{rfm_uk_full['Monetary'].max():,.2f}",
        ],
    }
)

rfm_validation_summary

Đoạn code dưới đây kiểm tra độc lập ba công thức RFM: số dòng phải bằng số khách UK, `Frequency` phải bằng số hóa đơn riêng biệt, và `Monetary` phải bằng tổng `TotalPrice` theo khách hàng.

In [ ]:
independent_check = (
    uk_df.groupby("CustomerID")
    .agg(
        FrequencyCheck=("InvoiceNo", "nunique"),
        MonetaryCheck=("TotalPrice", "sum"),
        LastPurchaseCheck=("InvoiceDate", "max"),
    )
    .reset_index()
)
independent_check["RecencyCheck"] = (
    snapshot_date - independent_check["LastPurchaseCheck"].dt.normalize()
).dt.days

rfm_check = rfm_uk_full.merge(independent_check, on="CustomerID", how="left")

validation_checks = pd.DataFrame(
    {
        "Check": [
            "RFM rows equal distinct UK customers",
            "Frequency equals distinct invoices",
            "Monetary equals grouped TotalPrice",
            "Recency uses snapshot date",
        ],
        "Passed": [
            len(rfm_uk_full) == uk_df["CustomerID"].nunique(),
            (rfm_check["Frequency"] == rfm_check["FrequencyCheck"]).all(),
            ((rfm_check["Monetary"] - rfm_check["MonetaryCheck"].round(2)).abs() < 0.01).all(),
            (rfm_check["Recency"] == rfm_check["RecencyCheck"]).all(),
        ],
    }
)

validation_checks

**Nhận xét:** nếu toàn bộ kiểm tra đều `True`, bảng RFM đã được tạo đúng công thức và có thể dùng cho bước khám phá RFM cũng như huấn luyện GMM.

## 6. Lưu file RFM

Sau khi kiểm tra xong, bảng RFM được lưu thành file mới để dùng cho phase sau. File cũ `rfm_uk.csv` không bị ghi đè.

Đoạn code dưới đây lưu bảng RFM full-period của UK thành `data/processed/rfm_uk_full.csv`.

In [ ]:
rfm_uk_full.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {OUTPUT_PATH} with {len(rfm_uk_full):,} customers.")